# ONNX generation loop + comparison vs torch

The autoregressive generation loop (greedy / beam) written out explicitly over the three ONNX graphs, then a side-by-side comparison against the torch checkpoint (`transformers .generate()`) on real line crops: exact-match rate, char similarity, latency, and visual samples.

Run on the box (needs `onnxruntime` for the loop; `torch`+`transformers` only for the reference).

In [1]:
import sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

here = Path.cwd()
ROOT = here if (here / "src").exists() else here.parent          # recognition/
sys.path.insert(0, str(ROOT / "onnx"))
from runtime_onnx import OnnxTrOCR

# ---- paths to set ----
ONNX_DIR = ROOT / "onnx_out"                                      # export_onnx.py output
CKPT     = ROOT / "outputs/trocr_small_bi_finetune_with_hwr200_cleaned/best"
IMAGES   = ROOT / "data/line_crops"                               # folder with line crops
N        = 30                                                      # crops to compare
DEVICE   = "cpu"                                                   # torch reference device

rec = OnnxTrOCR(ONNX_DIR)          # sessions + tokenizer + preprocess (int8=True for *_int8)
MAXLEN = rec.cfg["max_length"]

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp"}
paths = [p for p in sorted(IMAGES.rglob("*")) if p.suffix.lower() in IMG_EXT][:N]
crops = [Image.open(p).convert("RGB") for p in paths]
print(f"{len(crops)} crops | max_length {MAXLEN} | vocab {rec.cfg['vocab_size']}")

0 crops | max_length 128 | vocab 65000


In [6]:
import onnxruntime
onnxruntime.__version__

'1.23.2'

## Tokenizer — standalone usage
The service tokenizer is a single file, **`onnx_out/tokenizer.json`** (byte-level BPE, written by `export_onnx.py`), loaded by the light `tokenizers` lib — no transformers. Decode is the only thing generation needs; special token ids come from `service_config.json`, not from the tokenizer.

In [2]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(str(ONNX_DIR / "tokenizer.json"))   # the ONLY file it needs
# (rec.tok is this same object — OnnxTrOCR loads it the same way)

text = "Когда чувства оказываются сильнее голоса разума?"
enc = tok.encode(text)
print("tokens :", enc.tokens[:12], "...")
print("ids    :", enc.ids[:12], "...")

# decode is what the generation loop uses: list[int] token ids -> string
print("decode :", tok.decode(enc.ids, skip_special_tokens=True))
assert tok.decode(enc.ids, skip_special_tokens=True) == text   # byte-level BPE round-trips

# special ids used by generation come from service_config.json:
print("special ids:", {k: rec.cfg[k] for k in
      ("decoder_start_token_id", "eos_token_id", "pad_token_id")})
print("vocab size :", tok.get_vocab_size())

tokens : ['<s>', 'ÐļÐ¾Ð³Ð´Ð°', 'ĠÑĩÑĥÐ²ÑģÑĤÐ²Ð°', 'ĠÐ¾ÐºÐ°Ð·ÑĭÐ²Ð°ÑİÑĤÑģÑı', 'ĠÑģÐ¸Ð»ÑĮÐ½ÐµÐµ', 'ĠÐ³Ð¾Ð»Ð¾ÑģÐ°', 'ĠÑĢÐ°Ð·', 'ÑĥÐ¼Ð°', '?', '</s>'] ...
ids    : [0, 28228, 20740, 39385, 45286, 41431, 676, 5813, 35, 2] ...
decode : Когда чувства оказываются сильнее голоса разума?
special ids: {'decoder_start_token_id': 0, 'eos_token_id': 2, 'pad_token_id': 1}
vocab size : 65000


## The generation loop over the ONNX graphs

One function, both modes. Plumbing (feeding the graphs by input name) lives in `runtime_onnx` (`rec._first_step` / `rec._next_step`); the *loop logic* is all here:

- **greedy** (`num_beams=1`): whole batch at once, argmax each step, per-row EOS tracking;
- **beam**: first step on one row → take top-k tokens as initial beams → tile encoder states and KV cache to k rows; each step: total log-prob per (beam, token), keep the best k, **reorder the KV cache to the surviving beams**, park hypotheses that hit EOS (score normalised by length).

In [3]:
def log_softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=-1, keepdims=True))


def onnx_generate(rec, images, num_beams=1, max_length=None, length_penalty=1.0):
    """Recognize PIL line crops with the 3 ONNX graphs. num_beams=1 -> batched greedy;
    num_beams>1 -> beam search per image. Returns list[str]."""
    cfg = rec.cfg
    start, eos, pad = cfg["decoder_start_token_id"], cfg["eos_token_id"], cfg["pad_token_id"]
    max_len = int(max_length or cfg["max_length"])

    if num_beams == 1:                                        # ---- greedy, batched ----
        enc = rec.encode(rec.preprocess(images))
        B = enc.shape[0]
        logits, cache = rec._first_step(np.full((B, 1), start, np.int64), enc)
        seqs, done = [[] for _ in range(B)], np.zeros(B, bool)
        for _ in range(max_len - 1):
            nxt = logits[:, -1, :].argmax(-1)                 # most likely token per row
            nxt = np.where(done, pad, nxt)
            for b in range(B):
                if not done[b]:
                    if nxt[b] == eos: done[b] = True
                    else: seqs[b].append(int(nxt[b]))
            if done.all(): break
            logits, cache = rec._next_step(nxt.reshape(B, 1).astype(np.int64), enc, cache)
        return [rec.tok.decode(s, skip_special_tokens=True).strip() for s in seqs]

    out = []                                                  # ---- beam, per image ----
    for image in images:
        enc = rec.encode(rec.preprocess([image]))
        logits, cache = rec._first_step(np.array([[start]], np.int64), enc)
        lp = log_softmax(logits[0, -1, :].astype(np.float64))
        lp[eos] = -np.inf                                     # no empty hypothesis
        top = np.argsort(lp)[::-1][:num_beams]                # k initial beams
        scores, seqs = lp[top], [[int(t)] for t in top]
        enc_t = np.repeat(enc, num_beams, axis=0)             # tile enc states + KV to beams
        cache = {k: np.repeat(v, num_beams, axis=0) for k, v in cache.items()}
        finished, last = [], np.array(top, np.int64).reshape(-1, 1)
        for step in range(2, max_len):
            logits, cache = rec._next_step(last, enc_t, cache)
            lp = log_softmax(logits[:, -1, :].astype(np.float64))
            V = lp.shape[-1]
            total = (scores[:, None] + lp).ravel()            # joint score of (beam, token)
            new_seqs, new_scores, src, toks = [], [], [], []
            for cand in np.argsort(total)[::-1][: 2 * num_beams]:
                b, t = int(cand // V), int(cand % V)
                if t == eos:                                  # hypothesis complete
                    finished.append((total[cand] / (step ** length_penalty), seqs[b]))
                else:
                    new_seqs.append(seqs[b] + [t]); new_scores.append(total[cand])
                    src.append(b); toks.append(t)
                if len(new_seqs) == num_beams: break
            if not new_seqs: break
            idx = np.array(src)
            cache = {k: v[idx] for k, v in cache.items()}     # KV rows follow their beams
            seqs, scores = new_seqs, np.array(new_scores)
            last = np.array(toks, np.int64).reshape(-1, 1)
            if len(finished) >= num_beams and \
               scores.max() / (step ** length_penalty) <= min(f[0] for f in finished[:num_beams]):
                break                                          # no running beam can win
        if not finished:
            finished = [(s / (len(q) ** length_penalty), q) for s, q in zip(scores, seqs)]
        best = max(finished, key=lambda f: f[0])[1]
        out.append(rec.tok.decode(best, skip_special_tokens=True).strip())
    return out


# quick run
t0 = time.perf_counter(); onnx_greedy = onnx_generate(rec, crops, num_beams=1)
dt_g = (time.perf_counter() - t0) / len(crops)
t0 = time.perf_counter(); onnx_beam = onnx_generate(rec, crops, num_beams=4)
dt_b = (time.perf_counter() - t0) / len(crops)
print(f"onnx greedy {dt_g*1e3:.0f} ms/line | onnx beam-4 {dt_b*1e3:.0f} ms/line")
onnx_greedy[:3]

ValueError: need at least one array to stack

## Torch reference (`transformers .generate()`) on the same crops

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model = VisionEncoderDecoderModel.from_pretrained(str(CKPT)).eval().to(DEVICE)
proc = TrOCRProcessor.from_pretrained(str(CKPT))

def torch_generate(images, num_beams=1):
    out = []
    with torch.no_grad():
        for im in images:
            px = proc(images=im, return_tensors="pt").pixel_values.to(DEVICE)
            ids = model.generate(px, num_beams=num_beams, max_length=MAXLEN)
            out.append(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip())
    return out

t0 = time.perf_counter(); ref_greedy = torch_generate(crops, 1)
dt_rg = (time.perf_counter() - t0) / len(crops)
t0 = time.perf_counter(); ref_beam = torch_generate(crops, 4)
dt_rb = (time.perf_counter() - t0) / len(crops)
print(f"torch greedy {dt_rg*1e3:.0f} ms/line | torch beam-4 {dt_rb*1e3:.0f} ms/line")

## Comparison
Greedy is deterministic — ONNX should match torch (near-)exactly. Beam-4: score ties may break differently, judge by char similarity.

In [ ]:
try:
    from rapidfuzz.distance import Levenshtein
    sim = Levenshtein.normalized_similarity
except ImportError:
    sim = lambda a, b: 1.0 if a == b else 0.0

def compare(tag, onnx_texts, ref_texts, dt_onnx, dt_ref):
    same = sum(o == r for o, r in zip(onnx_texts, ref_texts))
    sims = [sim(o, r) for o, r in zip(onnx_texts, ref_texts)]
    print(f"[{tag}] exact {same}/{len(ref_texts)} | mean char-sim {np.mean(sims):.4f} | "
          f"onnx {dt_onnx*1e3:.0f} ms vs torch {dt_ref*1e3:.0f} ms per line")
    for p, o, r in zip(paths, onnx_texts, ref_texts):
        if o != r:
            print(f"  {p.name}\n    torch: {r}\n    onnx : {o}")

compare("greedy", onnx_greedy, ref_greedy, dt_g, dt_rg)
print()
compare("beam-4", onnx_beam, ref_beam, dt_b, dt_rb)

## Visual check — crop + both texts

In [4]:
k = min(6, len(crops))
fig, axes = plt.subplots(k, 1, figsize=(14, 1.6 * k))
for ax, im, o, r in zip(np.atleast_1d(axes), crops[:k], onnx_beam[:k], ref_beam[:k]):
    ax.imshow(im); ax.axis("off")
    mark = "==" if o == r else "!="
    ax.set_title(f"onnx: {o}   {mark}   torch: {r}", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

ValueError: Number of rows must be a positive integer, not 0

<Figure size 1400x0 with 0 Axes>